# Chapter 3: Enabling actions — Tool use

Companion notebook for *Build an AI Agent (From Scratch)*, Chapter 3. The cell layout follows the listings in the chapter 1:1.

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## 3.2 How LLMs use tools

### 3.2.1 How tool calling works

**Listing 3.1** Calculator tool definition schema

In [ ]:
calculator_tool_definition = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Perform basic arithmetic operations.",
        "parameters": {
            "type": "object",
            "properties": {
                "operator": {
                    "type": "string",
                    "description": "Arithmetic operation to perform",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "first_number": {
                    "type": "number",
                    "description": "First number for the calculation"
                },
                "second_number": {
                    "type": "number",
                    "description": "Second number for the calculation"
                }
            },
            "required": ["operator", "first_number", "second_number"],
        }
    }
}

**Listing 3.2** Calculator function implementation

In [ ]:
def calculator(operator: str, first_number: float, second_number: float):
    if operator == 'add':
        return first_number + second_number
    elif operator == 'subtract':
        return first_number - second_number
    elif operator == 'multiply':
        return first_number * second_number
    elif operator == 'divide':
        if second_number == 0:
            raise ValueError("Cannot divide by zero")
        return first_number / second_number
    else:
        raise ValueError(f"Unsupported operator: {operator}")

**Listing 3.3** Tool calling execution example

The first call has a question that doesn't need a tool; the second one does.

In [ ]:
import json
from litellm import completion

tools = [calculator_tool_definition]

response_without_tool = completion(
    model='gpt-5-mini',
    messages=[{"role": "user", "content": "What is the capital of South Korea?"}],
    tools=tools,
)
print("# Capital question doesn't need a tool call")
print(response_without_tool.choices[0].message.content)
print(response_without_tool.choices[0].message.tool_calls)

response_with_tool = completion(
    model='gpt-5-mini',
    messages=[{"role": "user", "content": "What is 1234 x 5678?"}],
    tools=tools,
)
print("\n# Multiplication question needs a tool call")
print(response_with_tool.choices[0].message.content)
print(response_with_tool.choices[0].message.tool_calls)

**Listing 3.4** Extract and execute tool from response

In [ ]:
ai_message = response_with_tool.choices[0].message
if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        if function_name == "calculator":
            result = calculator(**function_args)
            print(f"{function_name}({function_args}) -> {result}")

**Listing 3.5** Feed tool results back to LLM

We append the assistant's tool-call message, the `tool` role result, and let the model produce a final answer.

In [ ]:
messages = [{"role": "user", "content": "What is 1234 x 5678?"}]
ai_message = response_with_tool.choices[0].message
messages.append({                                # A: append the assistant's tool call
    "role": "assistant",
    "content": ai_message.content,
    "tool_calls": ai_message.tool_calls,
})
if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name  # B: parse + execute the tool
        function_args = json.loads(tool_call.function.arguments)
        if function_name == "calculator":
            result = calculator(**function_args)
            messages.append({                    # C: append the tool result
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result),
            })

final_response = completion(
    model='gpt-5-mini',
    messages=messages,
)
print("Messages:", messages)
print("Final Answer:", final_response.choices[0].message.content)

### 3.2.3 Guidelines for effective tool calling

Listings 3.6–3.11 are JSON schema illustrations rather than runnable Python. They appear here as reference for the patterns discussed in the chapter.

**Listing 3.6** Anti-pattern: vague tool definition

```json
{
  "name": "book",
  "description": "Book something",
  "parameters": {
    "type": "object",
    "properties": { "when": { "type": "string" } }
  }
}
```

**Listing 3.7** Improved: explicit tool definition

```json
{
  "name": "reserve_table",
  "description": "Create a restaurant table reservation.",
  "parameters": {
    "type": "object",
    "properties": {
      "restaurant_id": {
        "type": "string",
        "description": "Internal ID, e.g., rst_123"
      },
      "datetime": {
        "type": "string",
        "format": "date-time",
        "description": "ISO-8601 in the user's local time"
      },
      "party_size": {
        "type": "integer",
        "minimum": 1,
        "maximum": 20
      },
      "notes": {
        "type": "string",
        "description": "Allergies, occasion, etc. Optional."
      }
    },
    "required": ["restaurant_id", "datetime", "party_size"]
  }
}
```

**Listing 3.8** Anti-pattern: conflicting parameters

```json
{
  "name": "toggle_light",
  "parameters": {
    "type": "object",
    "properties": {
      "on":  { "type": "boolean" },
      "off": { "type": "boolean" }
    }
  }
}
```

**Listing 3.9** Improved: single enum parameter

```json
{
  "name": "toggle_light",
  "description": "Control the light state",
  "parameters": {
    "type": "object",
    "properties": {
      "state": {
        "type": "string",
        "enum": ["on", "off"],
        "description": "The desired state of the light"
      }
    },
    "required": ["state"]
  }
}
```

**Listing 3.10** Anti-pattern: model-supplied identifiers

```json
{
  "name": "submit_refund",
  "parameters": {
    "type": "object",
    "required": ["order_id", "reason"],
    "properties": {
      "order_id": {
        "type": "string",
        "description": "The order ID to refund"
      },
      "reason": {
        "type": "string",
        "description": "Reason for the refund"
      }
    }
  }
}
```

**Listing 3.11** Improved: decision-only parameters

```json
{
  "name": "submit_refund",
  "parameters": {
    "type": "object",
    "required": ["reason"],
    "properties": {
      "reason": { "type": "string" }
    }
  }
}
```

## 3.3 Building tools and tool definitions for LLMs

### 3.3.1 Implementing a web search tool

Install `tavily-python` and set `TAVILY_API_KEY` in your `.env` first.

**Listing 3.12** Basic web search function using Tavily

In [ ]:
import os
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()
tavily_client = TavilyClient(os.getenv("TAVILY_API_KEY"))

def search_web(query: str, max_results: int = 2) -> list:
    response = tavily_client.search(query, max_results=max_results)
    return response.get("results")

**Listing 3.13** Testing the web search function

In [ ]:
search_web("Kipchoge's marathon world record")

**Listing 3.14** Web search function with additional options

In [ ]:
def search_web(
    query: str,
    max_results: int = 2,
    topic: str = "general",
    time_range: str | None = None,
    country: str | None = None,
) -> list:
    response = tavily_client.search(
        query,
        max_results=max_results,
        topic=topic,
        time_range=time_range,
        country=country,
    )
    return response.get("results")

**Listing 3.15** Web search function with error handling

In [ ]:
def search_web(
    query: str,
    max_results: int = 5,
    topic: str = "general",
    time_range: str | None = None,
) -> list | str:
    """Search the web for the given query."""
    try:
        response = tavily_client.search(
            query,
            max_results=max_results,
            topic=topic,
            time_range=time_range,
        )
        return response.get("results")
    except Exception as e:
        return f"Error: Search failed - {e}"

### 3.3.2 Converting to tool definitions

**Listing 3.16** Extract tool metadata using `inspect`

In [ ]:
import inspect

def example_tool(input_1: str, input_2: int = 1):
    """docstring for example_tool"""
    return

print(f"function name: {example_tool.__name__}")
print(f"function docstring: {example_tool.__doc__}")
print(f"function signature: {inspect.signature(example_tool)}")

**Listing 3.17** Function to input schema converter

In [ ]:
def function_to_input_schema(func) -> dict:
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }
    try:
        signature = inspect.signature(func)                     # A
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {e}"
        )

    parameters = {}
    for param in signature.parameters.values():
        param_type = type_map.get(param.annotation, "string")    # B
        parameters[param.name] = {"type": param_type}            # B

    required = [
        param.name                                               # C
        for param in signature.parameters.values()               # C
        if param.default == inspect._empty                       # C
    ]

    return {
        "type": "object",
        "properties": parameters,
        "required": required,
    }

**Listing 3.18** Function to tool definition converter

In [ ]:
def format_tool_definition(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }


def function_to_tool_definition(func) -> dict:
    return format_tool_definition(
        func.__name__,
        func.__doc__ or "",
        function_to_input_schema(func),
    )


search_tool_definition = function_to_tool_definition(search_web)
print(search_tool_definition)

### 3.3.3 End-to-end tool execution

**Listing 3.19** Tool execution utility function

In [ ]:
def tool_execution(tool_box, tool_call):
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    tool_result = tool_box[function_name](**function_args)
    return tool_result

**Listing 3.20** Simple agent loop for tool execution

In [ ]:
from litellm import completion

def simple_agent_loop(system_prompt, question):
    tools = [search_web]
    tool_box = {tool.__name__: tool for tool in tools}
    tool_definitions = [function_to_tool_definition(tool) for tool in tools]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    while True:
        response = completion(
            model="gpt-5-mini",
            messages=messages,
            tools=tool_definitions,
        )
        assistant_message = response.choices[0].message
        if assistant_message.tool_calls:
            messages.append(assistant_message)
            for tool_call in assistant_message.tool_calls:
                tool_result = tool_execution(tool_box, tool_call)
                messages.append({
                    "role": "tool",
                    "content": str(tool_result),
                    "tool_call_id": tool_call.id,
                })
        else:
            return assistant_message.content

**Listing 3.21** Testing the agent loop

In [ ]:
system_prompt = """You are a helpful assistant.
Use the search tool when you need current information."""

result = simple_agent_loop(
    system_prompt,
    "Who won the 2025 Nobel Prize in Physics?",
)
print(result)

## 3.4 MCP: Standardizing tools

### 3.4.3 Understanding the MCP client

Install the MCP Python SDK first: `uv add mcp`. Section 3.4.2 in the chapter walks through running the official Tavily MCP server with `npx @modelcontextprotocol/inspector npx -y tavily-mcp@latest` (Node 22 LTS or later required).

**Listing 3.22** Connecting to an MCP server

In [ ]:
import asyncio
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from dotenv import load_dotenv

load_dotenv()

server_params = StdioServerParameters(
    command="npx",
    args=["-y", "tavily-mcp@latest"],
    env={
        "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY"),
    },
)

async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        # List available tools
        tools_result = await session.list_tools()
        print("Available tools:")
        for tool in tools_result.tools:
            print(f"  - {tool.name}: {tool.description[:60]}...")

        result = await session.call_tool(
            "tavily-search",
            arguments={"query": "2025 Nobel Physics"},
        )
        print("Search Result:")
        print(result.content)

**Listing 3.23** Converting MCP tools to OpenAI format

In [ ]:
def mcp_tools_to_openai_format(mcp_tools) -> list[dict]:
    """Convert MCP tool definitions to OpenAI tool format."""
    return [
        format_tool_definition(
            name=tool.name,
            description=tool.description,
            parameters=tool.inputSchema,
        )
        for tool in mcp_tools.tools
    ]

**Listing 3.24** Retrieving tools in OpenAI format

In [ ]:
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        # List available tools
        tools_result = await session.list_tools()
        openai_format_tools = mcp_tools_to_openai_format(tools_result)
        for tool in openai_format_tools:
            print(tool)

### 3.4.4 Hands-on: Implementing an MCP server

Install FastMCP first: `uv add fastmcp`. The next cell uses `%%writefile` to create the `tavily_mcp_server.py` script that we'll launch as a subprocess.

**Listing 3.25** Custom Tavily MCP server implementation

In [ ]:
%%writefile tavily_mcp_server.py
import os
from tavily import TavilyClient
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

tavily_client = TavilyClient(os.getenv("TAVILY_API_KEY"))

mcp = FastMCP("custom-tavily-search")


@mcp.tool()
def search_web(query: str, max_results: int = 5) -> str:
    """
    Search the web using Tavily API.

    Args:
        query: Search query string
        max_results: Maximum number of results to return (default: 5)

    Returns:
        Search results as formatted string
    """
    try:
        response = tavily_client.search(
            query,
            max_results=max_results,
        )
        results = response.get("results", [])
        return "\n\n".join(
            f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}"
            for r in results
        )
    except Exception as e:
        return f"Error searching web: {str(e)}"


if __name__ == "__main__":
    mcp.run(transport="stdio")

**Listing 3.26** Connecting to the custom MCP server

In [ ]:
server_params = StdioServerParameters(
    command="uv",
    args=["run", "tavily_mcp_server.py"],
    env={
        "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY"),
    },
)

async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        # List available tools
        tools_result = await session.list_tools()
        print("Available tools:")
        for tool in tools_result.tools:
            print(f"  - {tool.name}: {tool.description}")